# Setup and Your First Model Call

**What you will build:** a model-agnostic setup where one variable switches between Claude, GPT, Gemini, Llama and dozens more; your first LLM call; and a hands-on look at three things that shape the whole course — the **raw request format**, the API's **statelessness**, and **tokens**.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/01_setup_and_first_call.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

## One setup decision, explained

We need a way to call a model that is model-agnostic, durable (so this course doesn't rot every few months), and honest about the mechanics rather than hiding them behind a wrapper.

The choice: the official **`openai` SDK**, pointed at **[OpenRouter](https://openrouter.ai)**.

- The OpenAI *Chat Completions* format is the closest thing this field has to a standard — most providers speak it. Learn it once and it transfers everywhere.
- OpenRouter is a single gateway to every model. One key, one `base_url`, and you reach Claude, GPT, Gemini, Llama, DeepSeek and more by changing only a `MODEL` string. That is exactly the "let the reader pick the model" property we want.
- Fewer moving parts means less to break. The `openai` SDK and the Chat Completions schema are about as stable as anything here. The n8n course used OpenRouter too, so this is also continuity.

```{note}
You may have heard of **LiteLLM**, which also does model-agnostic calls. It's good, but it's an extra layer that moves faster than the base SDK. We keep it as an optional production add-on (see the appendix), not the teaching default. The pattern you learn here transfers to it in one import.
```

## Step 1 — Install the SDK

One small dependency. In a notebook, `!` runs a shell command.

In [ ]:
!pip install -q openai

## Step 2 — Get an API key

Create a key at **[openrouter.ai/keys](https://openrouter.ai/keys)** (it starts with `sk-or-`). OpenRouter offers free models — their id ends in `:free` — so you can do the whole course without a credit card.

```{tip}
Never paste an API key into a cell you might share. The cell below reads it with `getpass`, so it never appears in the notebook. In Colab you can instead store it once in the Secrets panel (the key icon in the left sidebar) as `OPENROUTER_API_KEY` and read it with `google.colab.userdata` — see the commented lines.
```

In [ ]:
import os, getpass

# Type it once per session (works everywhere):
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

# Colab alternative — read it from the Secrets panel instead of typing:
# from google.colab import userdata
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

## Step 3 — The `MODEL` variable

This one line decides which model powers everything below. Change it and re-run; nothing else needs to change. That is the model-agnostic promise, and it's the only knob you'll touch across the whole course.

```{note}
Model ids change over time. Check [openrouter.ai/models](https://openrouter.ai/models) for current names and prices. The `"vendor/model"` pattern is what's durable, not any one id.
```

In [ ]:
# Change THIS line to use a different model. Nothing else changes.
MODEL = "meta-llama/llama-3.3-70b-instruct"    # capable, cheap, supports tool-calling (needed later)

# Alternatives: paste any id from openrouter.ai/models (the pattern is "vendor/model"):
#   MODEL = "meta-llama/llama-3.3-70b-instruct:free"   # free tier, no card (rate-limited)
#   a current Anthropic Claude / OpenAI GPT / Google Gemini id for frontier quality (check the
#   models page for today's names -- specific ids get deprecated, the "vendor/model" shape does not)

print("Using model:", MODEL)

## Step 4 — Create the client

The official `OpenAI` client, aimed at OpenRouter's `base_url`. That one argument is what makes it model-agnostic — the same client class talks to every provider behind the gateway.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

## Step 5 — Your first call

A call is: send a list of **messages**, get one message back. Each message has a **role** and **content**.

| Role | Meaning |
|------|---------|
| `system` | Standing instructions or persona, applied to the whole conversation |
| `user` | What the human says |
| `assistant` | What the model replies (you append these yourself to build a conversation) |

We pass `temperature=0` to make the output as repeatable as this technology allows — more on that at the end.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant. Answer in one sentence."},
        {"role": "user",   "content": "What is an AI agent, in plain terms?"},
    ],
    temperature=0,
)

print(response.choices[0].message.content)

That is the whole primitive. Everything in this course — tools, memory, RAG, multi-agent — comes down to changing *what goes into that `messages` list* and *what you do with the message that comes back*.

## What actually went over the wire

The SDK is a thin convenience over a plain HTTPS request. Under the hood, Step 5 sent an HTTP `POST` to `https://openrouter.ai/api/v1/chat/completions` with a JSON body that is essentially the arguments you passed. Seeing the real request and response demystifies everything that follows — this is the level n8n's canvas hid from you.

The request body looks like this:

In [ ]:
import json

request_body = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a concise assistant. Answer in one sentence."},
        {"role": "user",   "content": "What is an AI agent, in plain terms?"},
    ],
    "temperature": 0,
}
print(json.dumps(request_body, indent=2))

And the response is JSON too. The SDK wraps it in an object, but you can dump the raw structure. Notice `choices` (the model can return more than one), `finish_reason` (why it stopped — this becomes important for tool calling in 0.3), and `usage` (the token counts you pay for).

In [ ]:
print(json.dumps(response.model_dump(), indent=2, default=str))

## The one property that explains "memory": statelessness

The API keeps **nothing** between calls. There is no session on the server. Each call is judged only on the messages you send in that call. Let's prove it — two independent calls, where the second asks about the first:

In [ ]:
# Call 1 — tell the model something.
client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Alex and I love hiking."}],
)

# Call 2 — a brand new, independent call. Ask it what we just said.
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name and what do I love?"}],
)
print(r.choices[0].message.content)

The model has no idea — because the two calls share nothing. This is not a limitation to fight; it's the design. If you want the model to "remember," *you* must re-send the earlier messages. That's all memory is.

## A conversation is just a growing list

So to hold a back-and-forth, append the model's reply and the next user turn to the same list, and call again. This is the exact mechanic behind "memory" in notebook 0.6 — there's no magic, just list-building.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful geography tutor."},
    {"role": "user",   "content": "What's the capital of Australia?"},
]

r1 = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
print("Assistant:", r1.choices[0].message.content)

# Append the reply, then ask a follow-up that only makes sense WITH the history:
messages.append({"role": "assistant", "content": r1.choices[0].message.content})
messages.append({"role": "user", "content": "And how many people live there?"})

r2 = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
print("Assistant:", r2.choices[0].message.content)

This time it resolved "there" correctly, because the earlier turns were in the list. The list *is* the memory — and because it grows every turn, it eventually bumps into the token budget, which is why 0.6 is a whole notebook.

## Tokens: what you send, and what you pay for

Models don't see characters or words; they see **tokens** (roughly 3–4 characters of English each). Two consequences you'll feel all course long:

- **Cost and speed scale with tokens.** Every call bills for input tokens + output tokens. The `usage` field tells you exactly how many.
- **The context window is finite.** Everything you send on a turn must fit a fixed token limit. Grow the message list forever and you'll hit it — hence memory management (0.6) and RAG (1.7).

Look at the usage from your multi-turn call:

In [ ]:
u = r2.usage
print("prompt (input) tokens:    ", u.prompt_tokens)
print("completion (output) tokens:", u.completion_tokens)
print("total tokens:              ", u.total_tokens)

## Determinism: the `temperature` knob

We passed `temperature=0` above. Temperature controls how much randomness the model uses when sampling the next token:

- `temperature=0` — take the most likely token each time. As repeatable as it gets; best for tools, extraction, tests.
- Higher values (up to ~1–2) — more variety and creativity, less repeatability.

```{warning}
Even at `temperature=0`, output is **not guaranteed identical** across runs — hardware and model updates introduce variation. Never build logic that assumes "same input, same output." This non-determinism is why we test agents with **evals** (1.6) rather than by running them once and eyeballing the result.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Swap the model.** Set `MODEL` to `"meta-llama/llama-3.3-70b-instruct:free"` (or a Claude/GPT id) and re-run Step 5. Same code, different model, different `response.model` in the raw dump.
2. **Feel the temperature.** Change `temperature` to `1.5`, set the prompt to `"Write a one-line slogan for a coffee shop."` and run it a few times. Then drop back to `0` and run a few more. Watch the variety collapse.
3. **Fix the memory demo.** In the statelessness cell, make the second call succeed by including the first message in its `messages` list. You've just implemented memory by hand.
::::

**What's next:** in **0.2** we go deeper on the `messages` format and add **structured output** — making the model return typed, validated data (a Python object you can trust) instead of free-form text.